# This is for creating views for the dashboard

##Expedition Outcomes

In [0]:
CREATE OR REPLACE VIEW himalaya.gold.vw_expedition_outcomes AS
SELECT
    f.expid AS `Expedition ID`,
    f.peakid AS `Peak ID`,
    p.mountain AS `Mountain`,
    p.height_m AS `Height (m)`,
    p.region AS `Region`,
    p.himalayan_range AS `Himalayan Range`,
    f.year AS `Year`,
    f.season AS `Season`,
    f.successes AS `Success`,
    f.termreason AS `Termination Reason`,
    f.deaths AS `Deaths`,
    f.total_members AS `Total Members`,
    f.highpoint AS `Highest Point`,
    f.oxygen_used AS `Oxygen Used`,
    w.temperature AS `Temperature (°C)`,
    w.rain AS `Rain (mm)`,
    w.snowfall AS `Snowfall (cm)`,
    w.snow_depth AS `Snow Depth (m)`,
    w.wind_speed AS `Wind Speed (km/h)`,
    w.weather_condition AS `Weather Condition`
FROM himalaya.gold.fact_expeditions f
LEFT JOIN himalaya.gold.dim_peaks p ON f.peakid = p.peakid
LEFT JOIN himalaya.gold.dim_weather w ON f.weather_id = w.weather_id;

##Deaths Detail

In [0]:
CREATE OR REPLACE VIEW himalaya.gold.vw_deaths_detail AS
SELECT
    d.name AS `Name`,
    d.death_date AS `Death Date`,
    d.peakid AS `Peak ID`,
    d.mountain AS `Mountain`,
    d.cause_of_death_category AS `Cause of Death`,
    d.expid AS `Expedition ID`,
    p.height_m AS `Height (m)`,
    p.region AS `Region`,
    p.himalayan_range AS `Himalayan Range`,
    w.temperature AS `Temperature (°C)`,
    w.rain AS `Rain (mm)`,
    w.snowfall AS `Snowfall (cm)`,
    w.snow_depth AS `Snow Depth (m)`,
    w.wind_speed AS `Wind Speed (km/h)`,
    w.weather_condition AS `Weather Condition`
FROM himalaya.gold.dim_deaths d
LEFT JOIN himalaya.gold.dim_peaks p ON d.peakid = p.peakid
LEFT JOIN himalaya.gold.dim_weather w ON CONCAT(d.peakid, '_', d.death_date) = w.weather_id;

##Peak Summary

In [0]:
CREATE OR REPLACE VIEW himalaya.gold.vw_peak_summary AS
SELECT
    p.peakid AS `Peak ID`,
    p.mountain AS `Mountain`,
    p.height_m AS `Height (m)`,
    p.region AS `Region`,
    p.himalayan_range AS `Himalayan Range`,
    p.first_ascent_year AS `First Ascent Year`,
    p.first_ascent_country AS `First Ascent Country`,
    p.first_ascent_members AS `First Ascent Members`,
    COUNT(DISTINCT f.expid) AS `Total Expeditions`,
    SUM(f.deaths) AS `Total Deaths`,
    ROUND(SUM(CASE WHEN f.successes = true THEN 1 ELSE 0 END) * 100.0 / COUNT(f.expid), 1) AS `Success Rate (%)`,
    ROUND(AVG(f.total_members), 1) AS `Avg Team Size`
FROM himalaya.gold.dim_peaks p
LEFT JOIN himalaya.gold.fact_expeditions f ON p.peakid = f.peakid
GROUP BY
    p.peakid, p.mountain, p.height_m, p.region, p.himalayan_range,
    p.first_ascent_year, p.first_ascent_country, p.first_ascent_members;

##Weather by Season

In [0]:
CREATE OR REPLACE VIEW himalaya.gold.vw_weather_by_season AS
SELECT
    w.peakid AS `Peak ID`,
    p.mountain AS `Mountain`,
    p.height_m AS `Height (m)`,
    CASE
        WHEN MONTH(w.date) IN (3, 4, 5) THEN 'Spring'
        WHEN MONTH(w.date) IN (6, 7, 8) THEN 'Summer'
        WHEN MONTH(w.date) IN (9, 10, 11) THEN 'Autumn'
        ELSE 'Winter'
    END AS `Season`,
    ROUND(AVG(w.temperature), 1) AS `Avg Temperature (°C)`,
    ROUND(AVG(w.wind_speed), 1) AS `Avg Wind Speed (km/h)`,
    ROUND(SUM(w.snowfall), 1) AS `Total Snowfall (cm)`,
    ROUND(SUM(w.rain), 1) AS `Total Rain (mm)`,
    ROUND(AVG(w.snow_depth), 1) AS `Avg Snow Depth (m)`
FROM himalaya.gold.dim_weather w
LEFT JOIN himalaya.gold.dim_peaks p ON w.peakid = p.peakid
GROUP BY
    w.peakid, p.mountain, p.height_m,
    CASE
        WHEN MONTH(w.date) IN (3, 4, 5) THEN 'Spring'
        WHEN MONTH(w.date) IN (6, 7, 8) THEN 'Summer'
        WHEN MONTH(w.date) IN (9, 10, 11) THEN 'Autumn'
        ELSE 'Winter'
    END;